# NeuroTutorSim — Bootstrap notebook (Colab)

**What this notebook is for.**

This notebook contains no *project logic*. It is an **execution shell**: its only
job is to take the code that lives on GitHub, mount it inside a Colab session with a GPU,
and connect it to Google Drive for persistent outputs.

The reason is §4.3 of the brief (*"every figure and table must be generated by a script"*).
If you write the logic inside the cells, the code lives in a session that dies after 12 hours
and the project becomes irreproducible.

**Golden rule:**

| What | Where it lives |
|---|---|
| Code, config, tests | GitHub |
| GPU execution | Colab (disposable) |
| Heavy outputs (parquet) | Google Drive |

This notebook is **only for Track A** (TRIBE inference, requires a GPU).
Track B (synthetic learners, Monte Carlo) runs on your laptop, not here.

---

## Step 0 — GPU check

If this cell says `NO GPU`, go to **Runtime → Change runtime type → T4 GPU**
and re-run it. Without a GPU, TRIBE won't start (or takes hours).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU — change runtime!"

---

## Step 1 — Clone the repo

The repo is **public**, so no GitHub token is needed for reading.
`git pull` at the end is there in case the session was still alive from before:
it re-aligns to the latest code without re-cloning.

In [ ]:
import os
REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
if not os.path.exists("/content/neurotutorsim"):
    !git clone {REPO} /content/neurotutorsim
%cd /content/neurotutorsim
!git pull --ff-only
!git log -1 --format="commit: %h | %s"

---

## Step 2 — Install dependencies from the lockfile

We use `uv` with `uv.lock`, so the Colab environment is **bit-for-bit identical** to the one
on your laptop (§4.3: *"pin all package versions"*).

`--extra tribe` installs torch/nilearn/nibabel, which are needed **only here**.
On your laptop, install without extras: no torch, no 2 GB download.

In [ ]:
!pip install -q uv
!uv sync --extra tribe --extra llm
# Make the uv environment visible to the Colab kernel
import sys
sys.path.insert(0, "/content/neurotutorsim/.venv/lib/python3.11/site-packages")
sys.path.insert(0, "/content/neurotutorsim")

---

## Step 3 — HuggingFace authentication

**Why a token is needed.** TRIBE is not self-contained: it uses **LLaMA 3.2-3B** as its text
encoder, and that model is *gated* by Meta.

The token does **not grant** access, it **carries** it. The gating is tied to your *account*.
If you don't yet have approval on `meta-llama/Llama-3.2-3B`, this cell will pass but
Step 5 will fail with `403 Gated repo`.

**Never paste the token here.** It goes in Colab Secrets:
🔑 icon in the left sidebar → `+ Add secret` → name `HF_TOKEN` → paste → enable
access for this notebook.

In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Authenticated as:", whoami()["name"])

### Preflight check: do I actually have access to LLaMA?

Better to find out now in 2 seconds than after a 20-minute download.

In [ ]:
from huggingface_hub import model_info
from huggingface_hub.utils import GatedRepoError
for repo in ["facebook/tribev2", "meta-llama/Llama-3.2-3B"]:
    try:
        model_info(repo, token=os.environ["HF_TOKEN"])
        print(f"OK      {repo}")
    except GatedRepoError:
        print(f"BLOCKED {repo}  -> request access at https://huggingface.co/{repo}")
    except Exception as e:
        print(f"ERROR   {repo}: {type(e).__name__}")

---

## Step 4 — Mount Drive

**The most important point of the notebook.**

Colab **has no persistent disk**. When the session dies (12h, or a disconnect, or
inactivity), everything in `/content/` disappears. If you write TRIBE's outputs there,
you lose them.

Drive is therefore your `data/processed/`. And the weights cache goes inside it: without it,
you re-download LLaMA (6 GB) every session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE  = "/content/drive/MyDrive/neurotutorsim"
CACHE = f"{BASE}/hf_cache"      # TRIBE + LLaMA weights: downloaded only once
OUT   = f"{BASE}/tribe_out"     # predictions: your deliverable D3
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = CACHE
print("cache :", CACHE)
print("output:", OUT)

---

## Step 5 — Load TRIBE

`tribev2` is not on PyPI: it must be cloned from Meta's repo.
We pin the **commit hash**, not `main`. §6.1 of the brief requires this explicitly
(*"record checkpoint, commit hash, hardware..."*): if Meta changes `main` between today and
submission, your results are no longer reproducible.

⚠️ Replace `TRIBE_COMMIT` with the real hash you read on GitHub.

In [ ]:
TRIBE_COMMIT = "main"  # <-- REPLACE with the commit hash
if not os.path.exists("/content/tribev2"):
    !git clone https://github.com/facebookresearch/tribev2.git /content/tribev2
    !cd /content/tribev2 && git checkout {TRIBE_COMMIT} && pip install -q -e .
from tribev2 import TribeModel
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE)
print("TRIBE loaded.")

---

## Step 6 — Smoke test

A single stimulus, to check that the chain holds end-to-end **before** launching
1,000 inferences.

This cell satisfies **decision gate #17** of the brief:
*"Do not launch full TRIBE inference until official examples are reproduced."*

Expected output: `(n_timesteps, ~20484)` — the vertices of the fsaverage5 mesh.

In [ ]:
text = (
    "Bayes theorem describes how to update a prior probability "
    "when new evidence becomes available. The posterior is proportional "
    "to the likelihood times the prior."
)
with open("/content/smoke.txt", "w") as f:
    f.write(text)
df = model.get_events_dataframe(text_path="/content/smoke.txt")
preds, segments = model.predict(events=df)
print("shape:", preds.shape)   # (time, vertices)
print("NaN  :", bool(preds.isnan().any()) if hasattr(preds, "isnan") else "n/a")

---

## Step 7 — The inference loop (skeleton)

From here on **the notebook must contain no logic**: it only calls functions from the repo.

Two non-negotiable properties, both required by the brief:

1. **Idempotency** (§4.3: *"never silently regenerate an existing item"*).
   The loop skips stimuli already present on Drive. If Colab disconnects you halfway,
   re-run it and it resumes from where it left off.

2. **Incremental writing.** Save *inside* the loop, never at the end. If you accumulate
   in memory and the session dies on the last stimulus, you lose everything.

We save at the **parcel** level (~1,000 parcels), not vertex (~20,000): 200 KB vs 4 MB
per stimulus. With ~1,000 stimuli that's 200 MB vs 4 GB — and free Drive gives you 15 in
total. Keep the vertex-level only for a subset, as a robustness check.

*(This is a deviation from §6.3 and must be agreed with the professor.)*

In [ ]:
# from src.tribe.inference import run_corpus    # <- to be implemented in the repo
# run_corpus(model, stimuli_csv="stimuli/stimuli.csv", out_dir=OUT, level="parcel")
import pandas as pd
from pathlib import Path
def run_corpus(model, stimuli_csv, out_dir):
    stimuli = pd.read_csv(stimuli_csv)
    for _, s in stimuli.iterrows():
        dest = Path(out_dir) / f"{s.stimulus_id}.parquet"
        if dest.exists():          # (1) idempotency
            continue
        df = model.get_events_dataframe(text_path=s.path)
        preds, _ = model.predict(events=df)
        # TODO: vertex -> parcel (src/tribe/aggregate.py, eq. 6-7 of the brief)
        # TODO: save metadata: commit, checkpoint, hardware, timestamp (§6.1)
        # to_parquet(preds, dest)  # (2) incremental writing
print("Skeleton ready. The logic goes in src/tribe/, not here.")